# Part 2, Topic 3: Voltage Glitching to Dump Memory (Hands-on Explorer)

---
NOTE: This lab references some (commercial) training material on [ChipWhisperer.io](https://www.ChipWhisperer.io). You can freely execute and use the lab per the open-source license (including using it in your own courses if you distribute similarly), but you must maintain notice about this source location. Consider joining our training course to enjoy the full experience.

---

**SUMMARY:** *Fault 2_2 showed that making a check lie can be hard - a narrow, drifting target. This lab glitches a different kind of bug: a response routine whose length is controlled by a variable that a fault can corrupt. You will find that this one is much easier to hit, and that on this kit the fault runs in the opposite direction from the lab's name - it cuts the response short rather than making it longer. Both directions come from the same root cause, and both matter for security.*

**LEARNING OUTCOMES:**

* Recognising a response-length bug as a fault-injection target
* Comparing how easy different kinds of bugs are to glitch
* Understanding how compiler-generated code can fail in a direction you did not expect


---
## Hands-on edition: how to use this notebook

Same format as Fault 2_1 and 2_2: cells marked **`>>> EDIT ME <<<`** are yours to change and re-run (**Shift+Enter**); cells marked **`RUN ONLY`** are helpers, run once. **My prediction** cells are for your notes.

**Before you start (hardware checklist)** - same kit as Fault 2_1 and 2_2:

1. ChipWhisperer-**Husky** plugged straight into a USB port (not through a hub).
2. **CW312T-SAM4S** target seated on the **CW313** board, ribbon connected.
3. The Husky **`Crowbar`** SMA cable connected to the CW313's second SMA socket.
4. No other notebook using the Husky.


## The firmware being tested

`main()` in `firmware/mcu/bootloader-glitch/bootloader.c` is a tiny 'upload an image' bootloader: send `p` followed by a hex-encoded message and a newline, and it:

1. decodes the hex into bytes,
2. 'decrypts' them (`decrypt_data()` in `decryption.c` - it is just a ROT-13 substitution; nothing secret),
3. and replies with a fixed 8-byte acknowledgement.

```c
ascii_idx = 0;
ascii_buffer[ascii_idx++] = 'r';
ascii_buffer[ascii_idx++] = '0';
ascii_buffer[ascii_idx++] = '\n';   /* x6 */
state = RESPOND;
...
if(state == RESPOND){
    trigger_high();
    for(i = 0; i < ascii_idx; i++)
        putch(ascii_buffer[i]);
    trigger_low();
}
```

The bug this lab targets: **`ascii_idx` controls how many bytes get sent back, and nothing re-checks it between being set and being used in that loop.** If a fault corrupts `ascii_idx` - or the loop's comparison against it - between those two points, the reply is the wrong length. Corrupt it *upward* past what was actually written and you leak whatever bytes happen to sit after it in memory (a classic over-read / 'memory dump'). Corrupt it *downward* and the reply is silently truncated - still a bug (a caller who expects 8 bytes gets fewer and may misinterpret what is missing), just a quieter one. Which direction you get depends on exactly how the compiler laid out `ascii_idx` and the loop, which is exactly the kind of thing that varies - this notebook finds out which direction happens on your own build.

In [ ]:
SCOPETYPE = 'OPENADC'
PLATFORM = 'CWHUSKY'
SS_VER = 'SS_VER_2_1'

In [ ]:
%%bash -s "$PLATFORM" "$SS_VER"
cd ../../../firmware/mcu/bootloader-glitch
make PLATFORM=$1 CRYPTO_TARGET=NONE -j SS_VER=$2

In [ ]:
%run "../../Setup_Scripts/Setup_Generic.ipynb"

In [ ]:
fw_path = "../../../firmware/mcu/bootloader-glitch/bootloader-{}.hex".format(PLATFORM)
cw.program_target(scope, prog, fw_path)

### Baseline: what does a normal upload look like?

**Do this:** run the next cell. It sends one hex-encoded message and prints the raw reply and the trigger length (`trig_count`) - how many ADC samples the `RESPOND` block's trigger stayed high for, which tells us how big an `ext_offset` range to search.

In [ ]:
def reboot_flush():
    reset_target(scope)
    target.flush()

scope.clock.adc_src = "clkgen_x1"
reboot_flush()
scope.adc.samples = 24000

PAYLOAD = "p516261276720736265747267206762206f686c207a76797821\n"   # a ROT13'd sentence, hex-encoded - the content does not matter
BASELINE = "r0\n\n\n\n\n\n"                                      # 8 bytes: the correct reply with no glitch

reboot_flush()
scope.arm()
target.write(PAYLOAD)
ret = scope.capture()
trig_count = scope.adc.trig_count
time.sleep(0.05)
output = target.read(timeout=2)
print("timed_out:", ret, " trig_count:", trig_count)
print("reply:", repr(output), " length:", len(output))
assert output == BASELINE, "flash/communication problem - fix this before continuing"

**You should see:** `reply: 'r0\n\n\n\n\n\n'  length: 8` and `trig_count` in the low thousands (2159 on our kit). If the assert fails, re-flash and re-run before doing anything else.

In [ ]:
if scope._is_husky:
    scope.vglitch_setup('hp', default_setup=False)
else:
    scope.vglitch_setup('both', default_setup=False)

In [ ]:
# ------------------------------------  RUN ONLY (do not edit)  ------------------------------------
import time, random
from collections import Counter
import matplotlib.pyplot as plt
import numpy as np

cw.set_all_log_levels(cw.logging.CRITICAL)
scope.adc.timeout = 0.3
if scope._is_husky:
    scope.adc.lo_gain_errors_disabled = True
    scope.adc.clip_errors_disabled = True

OUTCOME_COLORS = {"normal": "#c9d1d9", "reset": "#d9534f", "success": "#2e9e4f"}


def set_glitch(width, offset, ext_offset, repeat=1):
    if scope._is_husky:
        width, offset = int(round(width)), int(round(offset))
    scope.glitch.width = width
    scope.glitch.offset = offset
    scope.glitch.ext_offset = int(ext_offset)
    scope.glitch.repeat = int(repeat)


def fire_glitch(payload, width, offset, ext_offset, repeat=1):
    """Fire ONE glitch during one upload. Returns ("normal"|"reset"|"success", reply bytes or None).
    'success' here means the reply came back SHORTER than the 8-byte baseline - a truncated response."""
    set_glitch(width, offset, ext_offset, repeat)
    target.flush()
    if scope.adc.state:
        reboot_flush()
    scope.arm()
    target.write(payload)
    timed_out = scope.capture()
    scope.io.vglitch_reset()
    if timed_out:
        reboot_flush()
        return "reset", None
    time.sleep(0.05)
    reply = target.read(timeout=2)
    if not reply:
        reboot_flush()
        return "reset", reply
    outcome = "success" if len(reply) < len(BASELINE) else "normal"
    return outcome, reply


def fire_many(payload, width, offset, ext_offset, repeat=1, shots=20):
    tally = {"normal": 0, "reset": 0, "success": 0}
    replies = []
    cw.set_all_log_levels(cw.logging.CRITICAL)
    for _ in range(shots):
        outcome, reply = fire_glitch(payload, width, offset, ext_offset, repeat)
        tally[outcome] += 1
        if outcome == "success":
            replies.append(reply)
    return tally, replies


def scan(payload, points, shots):
    tallies = []
    for p in points:
        t, _ = fire_many(payload, *p, shots=shots)
        tallies.append(t)
        print(".", end="", flush=True)
    print()
    return tallies


def plot_tallies(labels, tallies, xlabel, title):
    fig, ax = plt.subplots(figsize=(9, 3.4))
    bottom = np.zeros(len(labels))
    for name in ("success", "reset", "normal"):
        share = np.array([100.0 * t[name] / max(1, sum(t.values())) for t in tallies])
        ax.bar(range(len(labels)), share, bottom=bottom, color=OUTCOME_COLORS[name], label=name, width=0.85)
        bottom += share
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels([str(l) for l in labels], rotation=90 if len(labels) > 12 else 0, fontsize=8)
    ax.set_xlabel(xlabel); ax.set_ylabel("% of shots"); ax.set_title(title); ax.set_ylim(0, 100)
    ax.legend(loc="center left", bbox_to_anchor=(1.0, 0.5), fontsize=8)
    plt.tight_layout(); plt.show()


def recover():
    scope.io.vglitch_reset()
    reboot_flush()
    scope.io.glitch_hp, scope.io.glitch_lp = True, False
    scope.glitch.repeat = 1
    print("target rebooted, glitch hardware restored")


print("helpers ready: fire_glitch, fire_many, scan, plot_tallies, recover")

### Experiment 1: try Fault 2_1/2_2's settings here too

**Goal:** by now you should expect the answer - but check it anyway. Does a setting that worked for the counting loop or the password check do anything to this response routine?

**Do this:** run the cell as it is.

In [ ]:
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>  EDIT ME  <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
WIDTH      = 1770
OFFSET     = 2670
EXT_OFFSET = 1
REPEAT     = 1
SHOTS      = 15
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

t0 = time.time()
tally, replies = fire_many(PAYLOAD, WIDTH, OFFSET, EXT_OFFSET, REPEAT, SHOTS)
print("width={} offset={} ext_offset={}   |   {} shots in {:.1f} s".format(WIDTH, OFFSET, EXT_OFFSET, SHOTS, time.time() - t0))
for name in ("normal", "reset", "success"):
    print("  {:8s} {:3d}  {}".format(name, tally[name], "#" * tally[name]))

**What you should see on our kit:** all `normal` - `ext_offset=1` is early in this routine's own trigger window (which starts when the upload begins, not when the reply is sent), and this width is too gentle to reach this firmware's own crash threshold anyway. Nothing happens here; the interesting part of this firmware's trigger window is much later.

**My prediction (Experiment 1):** _double-click and write here_

**What I saw:** _..._

### Experiment 2: find this firmware's own crash edge

**Goal:** scan width at a fixed offset and ext_offset, the way Fault 2_1 and 2_2 did, but somewhere later in the (much longer) trigger window - try `ext_offset` around 1500-1600, a point roughly two-thirds of the way through `trig_count`.

**Do this:** run the cell as it is.

In [ ]:
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>  EDIT ME  <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
OFFSET      = 2841
EXT_OFFSET  = 1559
WIDTH_START = 1600
WIDTH_STOP  = 2150
WIDTH_STEP  = 25
SHOTS       = 6
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

widths = list(range(WIDTH_START, WIDTH_STOP + 1, WIDTH_STEP))
t0 = time.time()
tallies = scan(PAYLOAD, [(w, OFFSET, EXT_OFFSET) for w in widths], SHOTS)
print("done in {:.0f} s".format(time.time() - t0))
plot_tallies(widths, tallies, "WIDTH", "offset={}  ext_offset={}".format(OFFSET, EXT_OFFSET))

**What you should see on our kit - and it is a much easier edge than Fault 2_2's:** at this exact `ext_offset`, width 1775 and below was `normal`; 1800-1825 a transition; and **1850 and above gave `success` every single time**, up to 2125, with every reply shortened to the same 6 bytes (`'r0\n\n\n\n'`). That is how clean this edge can look at one instant.

**GOOD TO KNOW - this moves too, like Fault 2_2's settings.** Re-running the exact same cell roughly twenty minutes later in the same session, `EXT_OFFSET=1559` had gone completely quiet - `normal` at every single width from 1600 to 2125, no `success` at all. Experiment 3 (next) found a *different* ext_offset nearby that had picked up the effect instead, and scanning width there again showed the same kind of edge, just less sharp - a mix of `normal`/`success`/occasional `reset` above about 1825 rather than a clean 100%. **The lesson is the same as Fault 2_2's: don't treat any single `(width, offset, ext_offset)` as permanent.** If a setting that worked goes quiet, look at its neighbours (especially in `ext_offset`) before assuming the effect is gone.

**My prediction (Experiment 2):** _double-click and write here_

**What I saw (where is the edge? how reliable is `success` past it?):** _..._

### Experiment 3: how much of the trigger window is like this?

**Goal:** Experiment 2 picked one `ext_offset` almost at random. Is the whole trigger window this forgiving, or only parts of it?

**Do this:** run the cell as it is - a reliably-strong width, scanned across the whole 0 to `trig_count` range in big steps.

In [ ]:
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>  EDIT ME  <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
WIDTH      = 1900
OFFSET     = 2841
EXT_STEP   = 150
SHOTS      = 5
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

exts = list(range(0, trig_count, EXT_STEP))
t0 = time.time()
tallies = scan(PAYLOAD, [(WIDTH, OFFSET, e) for e in exts], SHOTS)
print("done in {:.0f} s".format(time.time() - t0))
plot_tallies(exts, tallies, "EXT_OFFSET", "width={}  offset={}".format(WIDTH, OFFSET))
good = [e for e, t in zip(exts, tallies) if t["success"]]
print("ext_offset values with at least one success:", good)

**What you should see on our kit:** mostly `normal`, with `success` showing up at a handful of specific moments scattered through the window (around 750, 1350, 1800 and 2100 in one of our runs) - roughly one good moment every few hundred cycles, not a single pinpoint and not the whole window either. That is a much denser 'good' region than Fault 2_2 ever found for its password bypass: here, a plain scan over width and ext_offset, with no need for a blind random search, turns up several working settings in well under a minute.

**My prediction (Experiment 3):** _double-click and write here_

**What I saw (roughly how often does a good moment occur?):** _..._

### Experiment 4: reliability and payload-independence

**Goal:** confirm one of your settings is reliably reproducible, and check whether the uploaded message itself makes any difference - this fault corrupts a loop bound, not the message's content, so it should not.

**Do this:** run the cell at one of your settings from Experiment 2 or 3, with two very different payloads.

In [ ]:
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>  EDIT ME  <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
WIDTH, OFFSET, EXT_OFFSET = 1900, 2841, 1559
SHOTS = 20
PAYLOAD_A = PAYLOAD                       # the hex-encoded sentence from the baseline
PAYLOAD_B = "p" + "41" * 26 + "\n"         # the same length, but every byte is 0x41 ('A')
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

t0 = time.time()
for name, p in (("original message", PAYLOAD_A), ("all-A's message", PAYLOAD_B)):
    tally, replies = fire_many(p, WIDTH, OFFSET, EXT_OFFSET, shots=SHOTS)
    lens = sorted(set(len(r) for r in replies))
    print("{:18s} {}   reply lengths seen on success: {}".format(name, tally, lens))
print("done in {:.0f} s".format(time.time() - t0))

**What you should see on our kit:** both payloads behave almost identically - in one run, 14/20 and 12/20 successes, close enough to call 'the same rate' given how much these settings drift on their own (see Experiment 2) - and **every single success truncated to the same length** (6 bytes) regardless of which payload was sent. That confirms it: the fault is corrupting a loop bound or counter, not reading back part of the message you sent. Try your own setting from Experiment 3 - different settings can truncate to different lengths (we also saw replies of 1, 2, 3, 4, 5 and 7 bytes at other settings), but a *given* setting keeps giving the *same* length as long as it keeps working at all.

**My prediction (Experiment 4):** _double-click and write here_

**What I saw:** _..._

### Experiment 5: which direction does it fail in - and why does that matter?

**Goal:** this lab is named 'memory dump' because the intended bug is an **over-read**: a fault that makes `ascii_idx` *too big*, so the reply loop sends extra bytes that were never meant to go out - memory you did not upload, potentially including other data sitting nearby in SRAM. Check which direction your kit actually goes.

In [ ]:
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>  EDIT ME  <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
WIDTH_RANGE      = (1500, 2100)
OFFSET_RANGE     = (1500, 3500)
TIME_BUDGET_S    = 120
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

random.seed()
cw.set_all_log_levels(cw.logging.CRITICAL)
t0, attempts, longest = time.time(), 0, (len(BASELINE), None)
while time.time() - t0 < TIME_BUDGET_S:
    w = random.randint(*WIDTH_RANGE)
    o = random.randint(*OFFSET_RANGE)
    e = random.randint(0, trig_count - 1)
    outcome, reply = fire_glitch(PAYLOAD, w, o, e)
    attempts += 1
    if reply and len(reply) > longest[0]:
        longest = (len(reply), (w, o, e, reply))
        print("LONGER than baseline! width={} offset={} ext_offset={}  reply={!r}".format(w, o, e, reply))
cw.set_all_log_levels(cw.logging.WARNING)
print("{} attempts in {:.0f} s. Longest reply seen: {} bytes {}".format(
    attempts, time.time() - t0, longest[0], "(longer than baseline!)" if longest[0] > len(BASELINE) else "(baseline length or shorter, every time)"))

**What you should see on our kit:** in every search we ran (several thousand attempts in total, across a wide box), **the reply was never longer than the 8-byte baseline.** The fault only ever truncated it. That is the real, honest finding on this build: the same root cause (a fault hitting `ascii_idx` or the loop that uses it) produced an under-read here, not the over-read the lab's name promises. Both are real bugs - a silent truncation can be just as damaging in a different protocol, where a caller trusts a length field and a short reply means something was quietly dropped - but neither one is 'better' or 'worse' as a finding to report: **recording which direction actually happened, honestly, is more useful than assuming the dramatic one did.** If your own kit or a different compiler ever does produce a longer-than-baseline reply, that is the over-read this lab was named for - write down the exact setting and what extra bytes came back.

**My prediction (Experiment 5):** _double-click and write here_

**What I saw (did you ever see a reply longer than 8 bytes?):** _..._

## Challenges (optional)

1. **Map the full truncation spectrum.** Collect every distinct reply length you can find across several settings. Is there a pattern to which settings give which length?
2. **Compare search difficulty properly.** Time how long it takes a plain width-then-ext_offset scan (like Experiments 2 and 3) to find a working setting here, versus how long Fault 2_2's blind random search needed. Which bug was easier to glitch, and by roughly how much?
3. **Does `repeat` change the truncation length?** Try `REPEAT = 2` or `3` at a working setting.
4. **A longer payload.** Upload a message close to `ASCII_BUFLEN` (80 hex characters) and repeat Experiment 2. Does the crash edge or the truncated length change?

## If something goes wrong

| What you see | Most likely cause | What to do |
|---|---|---|
| The baseline assert fails | Firmware did not flash, or a stale serial buffer | Re-run the build/flash cells; re-run `reboot_flush()` |
| Every shot is `normal`, never `reset` or `success`, even at a huge width | The Crowbar SMA cable is not connected | Fit the cable, run the setup cells again |
| Every shot is `reset` | You are above the crash threshold for this width | Lower `WIDTH` a little |
| `target.read()` returns nothing at all | The target may be stuck mid-reply | Run `recover()`, then `reboot_flush()` |
| `OSError: Could not find ChipWhisperer` | Husky unplugged, or another notebook still holds it | Plug it in; shut down every other notebook; restart this kernel |
| `USBErrorPipe` / `Unknown hwInfoVer` | The Husky's USB link or FPGA stopped answering | Unplug and replug the Husky directly; restart the kernel |
| A cell was interrupted (Stop button) | Target or glitch hardware may be left in an odd state | Run `recover()` in a new cell |


## Finish: switch the glitch hardware off

**Always run this before closing the notebook.**

In [ ]:
scope.io.glitch_hp = False
scope.io.glitch_lp = False
scope.glitch.enabled = False
scope.dis()
target.dis()